In [10]:
import pandas as pd
import re
from nrclex import NRCLex
from sentence_transformers import SentenceTransformer, util

# Dummy dataset containing original and emotion-shifted sentences
data = {
    'original': [
        'The devastating storm destroyed the house.',
        'I am crying about the terrible test results.',
        'He walked slowly and miserably to the car.',
        'His insults were like a dagger to my heart.'
    ],
    'transformed': [
        'The beautiful storm blessed the house.',
        'I am cheering about the amazing test results.',
        'He walked quickly and happily to the car.',
        'His compliments were like a ray of sunshine to my heart.'
    ]
}

df = pd.DataFrame(data)
df.head()

,original,transformed
0,The devastating storm destroyed the house.,The beautiful storm blessed the house.
1,I am crying about the terrible test results.,I am cheering about the amazing test results.
2,He walked slowly and miserably to the car.,He walked quickly and happily to the car.
3,His insults were like a dagger to my heart.,His compliments were like a ray of sunshine to...


In [11]:
TARGET_EMOTIONS = {
    'fear', 'anger', 'anticipation', 'trust', 'surprise',
    'negative', 'positive', 'sadness', 'disgust', 'joy'
}


def is_emotion_token(token, target_emotions=TARGET_EMOTIONS):
    text = token.lower()

    # Skip tokens that are not words
    if not re.search(r"[a-z]", text):
        return False

    emo = NRCLex()
    emo.load_token_list([text])

    # affect_dict maps matched words to their NRC emotion labels
    if not getattr(emo, "affect_dict", {}):
        return False

    labels = set(emo.affect_dict.get(text, []))
    return bool(labels & target_emotions)


def mask_emotion_words(text):
    tokens = re.findall(r"\b[\w']+\b", text)
    masked_tokens = [token for token in tokens if not is_emotion_token(token)]
    return " ".join(masked_tokens)


df['masked_original'] = df['original'].apply(mask_emotion_words)
df['masked_transformed'] = df['transformed'].apply(mask_emotion_words)

print("Text masking complete.")

Text masking complete.


In [12]:
# Initialize the Sentence Transformer
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode only the masked sentences
emb_orig_masked = model.encode(df['masked_original'].tolist(), convert_to_tensor=True)
emb_trans_masked = model.encode(df['masked_transformed'].tolist(), convert_to_tensor=True)

emb_orig = model.encode(df['original'].tolist(), convert_to_tensor=True)
emb_trans = model.encode(df['transformed'].tolist(), convert_to_tensor=True)

# Calculate cosine similarities
cosine_scores_masked = util.cos_sim(emb_orig_masked, emb_trans_masked)
cosine_scores = util.cos_sim(emb_orig, emb_trans)

# Extract the diagonal (row-by-row similarity)
df['content_preservation_score'] = [cosine_scores[i][i].item() for i in range(len(df))]
df['content_preservation_score_masked'] = [cosine_scores_masked[i][i].item() for i in range(len(df))]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# Display the results. Notice how the masked scores stay focused on
# content words after removing tokens flagged by the NRC emotion lexicon.
display(df[
    [
        'original',
        'transformed',
        'masked_original',
        'masked_transformed',
        'content_preservation_score',
        'content_preservation_score_masked'
    ]
])

,original,transformed,masked_original,masked_transformed,content_preservation_score,content_preservation_score_masked
0,The devastating storm destroyed the house.,The beautiful storm blessed the house.,The the house,The the house,0.663852,1.000000
1,I am crying about the terrible test results.,I am cheering about the amazing test results.,I am about the test results,I am about the amazing test results,0.727604,0.838725
2,He walked slowly and miserably to the car.,He walked quickly and happily to the car.,He walked slowly and to the car,He walked quickly and to the car,0.701062,0.912201
3,His insults were like a dagger to my heart.,His compliments were like a ray of sunshine to...,His insults were like a to my heart,His compliments were like a ray of to my heart,0.515442,0.648189
